# Certified ARTE session to parameterized Polars backtest objective

Fourth notebook: real data, default **2026-08-18, 04:00–09:30 America/New_York**
(19,800 one-second execution slots). This is an approximate research environment,
not production broker equivalence. No search is performed here.

`prepare_session()` fetches/certifies once; `prepare_policy()` compiles parameterized
native Polars expressions; `evaluate(theta, prepared)` returns a scalar objective
and compact accounting evidence. Changing theta never queries the database.

Performance design: use persisted 1 s ARTE rollups, project only needed columns,
bound fetch chunks by measured row count and ticker count, cache immutable chunks
under the runtime root, and keep sparse rows rather than a timeÃ—universe Cartesian
grid. Precompute stateless rules over the whole session, then sequence actual
account transitions causally. The hot loop uses Polars expressions across active
positions/pending orders and current candidates; it does not loop over tickers.
The old named feature/market contract is projected into flat scalar columns here;
no redundant feature tensor is materialized for an expression-only strategy.
All session time slots are represented in accounting, even when no events occur.

ARTE bars are sparse. A *certified absent* interval means no eligible bar update,
not broken coverage: no fill, no new signal, carry the last valid mark and report
its age. Invalid price rows also cannot fill. Technical rows exist only for valid-price
bars; missing indicators on invalid-price rows are represented as null, while
missing indicators on valid-price bars reject the fetch. Certificate/hash failures stop the
run. The previous dummy notebook required dense coverage; that assumption changes
explicitly here to match the real producer's sparse contract.


In [ ]:
import os, sys, json, math, hashlib, sqlite3
from pathlib import Path
from datetime import date, datetime, time, timezone
from zoneinfo import ZoneInfo
from contextlib import closing
from dataclasses import dataclass, asdict
from time import perf_counter
# Small causal transitions are latency-bound; set before importing Polars.
os.environ.setdefault('POLARS_MAX_THREADS','1')
import polars as pl

os.environ['PYTHONDONTWRITEBYTECODE']='1'; sys.dont_write_bytecode=True
# Run from the repository or this notebook's directory; no machine-specific source copy.
REPO=next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'research/rl_trading').is_dir())
if str(REPO) not in sys.path: sys.path.insert(0,str(REPO))
from research.mlops.env import load_env_files, discover_env_files
from research.rl_trading.v1 import arte_sql as sql, arte_source
from research.rl_trading.v1.bracket_source import broker_attempts, assert_liquidity_storage
from scripts.clickhouse.install_market_day_certificate_layout import workstation_clickhouse_url
load_env_files(discover_env_files(REPO),verbose=False)
# Managed workstation listener. Explicit environment configuration takes precedence.
if not any(os.environ.get(k) for k in ('CLICKHOUSE_URL','REAL_LIVE_CLICKHOUSE_WRITE_URL',
                                      'TD__DATABASE__CLICKHOUSE__ENDPOINT_URL')):
    os.environ['CLICKHOUSE_URL']=os.environ.get('QMD_CLICKHOUSE_URL') or workstation_clickhouse_url()
for dst,src in [('CLICKHOUSE_USER','QMD_CLICKHOUSE_USER'),('CLICKHOUSE_PASSWORD','QMD_CLICKHOUSE_PASSWORD')]:
    if os.environ.get(src): os.environ.setdefault(dst,os.environ[src])

RUNTIME=Path('D:/TradingML/runtimes/vectorized_backtest/arte_polars_v1')
if not RUNTIME.parent.is_dir(): raise FileNotFoundError('Required runtime root unavailable')
RUNTIME.mkdir(exist_ok=True)
REMOTE_RUNTIME=Path('//DESKTOP-SAAI85T/Workstation-D/TradingML/runtimes')
BUILD_ID='1521ba7702a9ee0783916f706f4885a24a3f32a91630b04ff738a90e65bc9dd5'
MANIFEST=REMOTE_RUNTIME/'market-day'/f'{BUILD_ID}.json'
LEDGER=REMOTE_RUNTIME/'build-ledger-v2.sqlite3'
DAY=date(2026,8,18); NY=ZoneInfo('America/New_York')
START=datetime.combine(DAY,time(4),NY); END=datetime.combine(DAY,time(9,30),NY)
TICKERS=None  # None means the complete pinned tradable population; never top-N silently.
MAX_FETCH_ROWS=150_000; MAX_FETCH_TICKERS=512
MAX_PREPARED_GIB=2.0
INITIAL_CASH=100_000.0
PARTICIPATION=0.10  # Fixed fill-model parameter, not a strategy-search parameter.
FEE_BPS=1.0        # Fixed proportional fee per filled side, no spread/impact/queue model.
ENVIRONMENT_VERSION='arte-polars-sparse-vwap-1s-v1'
pl.Config.set_tbl_rows(12)
print({'day':str(DAY),'start':START.isoformat(),'end':END.isoformat(),
       'time_slots':int((END-START).total_seconds()),'polars':pl.__version__,'polars_threads':pl.thread_pool_size()})


## Authority, projection and chunk boundaries

`arte.bars_v1` at `resolution_ms=1000` already includes `execution_volume` and
`execution_notional`, derived by rollup from `arte.liquidity_100ms_v1`. Use their
interval ratio, never cumulative-session `execution_vwap`. This avoids fetching
and grouping ten times as many liquidity rows. `arte.indicators_v1` supplies
causal completed MACD line/signal. Quote/price-level execution is intentionally
outside this approximate fill model.

Reuse existing manifest/SQLite build certification and historical listing IDs.
Check storage and the full saved row count/key uniqueness/content hash of each
selected bars/technical unit before fetching. The broker unit must also be
certified, although its raw rows are not transferred. These are read-only queries.
Chunking by listing groups retains complete feature histories across time; measured
counts keep each response below the reader's 700,000-row cap. Zero-row listings
remain in the certified population and are reported; they are not rejected assets.
A single listing cannot exceed 19,800 rows for this session. Successful chunks have atomic files and a hash-bound progress checkpoint; interrupted
fetches revalidate the source and reuse matching chunks. A completed cache is sealed.
No concurrency is
needed initially: one bounded persistent reader avoids loading the shared server.


In [ ]:
DATA_SCHEMA={'ticker':pl.String,'second':pl.Int64,'close':pl.Float64,
             'price_valid':pl.Int64,'volume':pl.Float64,'notional':pl.Float64,
             'macd':pl.Float64,'signal':pl.Float64,'indicator_seen':pl.Int64}

def file_hash(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()

def scope_sql(source, day, tickers, stage):
    pairs=','.join(f'({sql.literal(t)},toUUID({sql.literal(source["units"][str(day)][t][stage]["attempt_id"])}))'
                   for t in tickers)
    return (f'build_id={sql.literal(source["build_id"])} AND session_date=toDate({sql.literal(day)}) '
            f'AND (ticker,attempt_id) IN ({pairs})')

def certify_units(reader,source,day,tickers):
    # Native server reductions validate entire unit certificates, not just the requested prefix.
    for offset in range(0,len(tickers),MAX_FETCH_TICKERS):
        group=tickers[offset:offset+MAX_FETCH_TICKERS]
        for stage,table in arte_source.TABLES.items():
            rows=sql.query(reader,f'SELECT ticker,count() AS n,uniqExact((resolution_ms,bucket_index)) AS u,'
                f'sum(cityHash64(tuple(*))) AS hash FROM arte.{table} '
                f'WHERE {scope_sql(source,day,group,stage)} GROUP BY ticker')
            actual={r['ticker']:r for r in rows}
            for ticker in group:  # Certificate comparison only, never an execution calculation.
                saved=source['units'][str(day)][ticker][stage]
                r=actual.get(ticker,{'n':0,'u':0,'hash':0})
                if int(r['n'])!=saved['output_rows'] or int(r['u'])!=int(r['n']) or str(r['hash'])!=saved['output_hash']:
                    raise ValueError(f'Persisted {stage} certificate mismatch: {ticker}')
        print({'certified_units':min(offset+len(group),len(tickers)),'total':len(tickers)},flush=True)

def prepare_session(start=START,end=END,tickers=TICKERS):
    if start.tzinfo is None or end.tzinfo is None or start.astimezone(NY).date()!=end.astimezone(NY).date() or end<=start:
        raise ValueError('Require increasing timezone-aware bounds in one session')
    day=start.astimezone(NY).date()
    origin=datetime.combine(day,time.min,NY)
    first=int((start.astimezone(timezone.utc)-origin.astimezone(timezone.utc)).total_seconds())
    last=int((end.astimezone(timezone.utc)-origin.astimezone(timezone.utc)).total_seconds())
    if not (14400<=first<last<=72000) or any(x.microsecond for x in (start,end)):
        raise ValueError('Bounds must be whole seconds within 04:00–20:00 New York')
    started=perf_counter()
    source=arte_source.load_build(MANIFEST,LEDGER,[day],tickers)
    names=sorted(source['units'][str(day)])
    attempts=broker_attempts(source,LEDGER,day,set(names))
    # Cache identity includes attempt/provenance and exact requested boundaries.
    signature={'version':ENVIRONMENT_VERSION,'source':source,'broker_attempts':attempts,
               'first':first,'last':last,'manifest_sha256':file_hash(MANIFEST)}
    key=hashlib.sha256(json.dumps(signature,sort_keys=True,default=str).encode()).hexdigest()
    cache=RUNTIME/key; cache.mkdir(exist_ok=True)
    cache_manifest=cache/'complete.json'
    with closing(sql.ArteReader(threads=2)) as reader:
        arte_source.storage_check(reader); assert_liquidity_storage(reader)
        members,population_proof=arte_source.population(reader,source,day)
        identities=pl.DataFrame(members).select('ticker',pl.col('listing_id').cast(pl.String))
        if cache_manifest.is_file():
            saved=json.loads(cache_manifest.read_text())
            if saved['key']!=key: raise ValueError('Wrong cached source identity')
            # Completed cache is sealed by per-file SHA256; never silently accept partial chunks.
            if sum(item['rows'] for item in saved['chunks'])*160/(1024**3)>MAX_PREPARED_GIB:
                raise MemoryError('Cached session exceeds prepared-memory guard')
            for item in saved['chunks']:
                if file_hash(cache/item['file'])!=item['sha256']: raise ValueError('Corrupted cached chunk')
            market=pl.concat([pl.read_parquet(cache/item['file']) for item in saved['chunks']])
            print({'cache':'verified_complete','rows':market.height},flush=True)
        else:
            certify_units(reader,source,day,names)
            counts=[]
            for offset in range(0,len(names),MAX_FETCH_TICKERS):
                group=names[offset:offset+MAX_FETCH_TICKERS]
                counts.extend(sql.query(reader,f'SELECT ticker,count() AS n FROM arte.bars_v1 '
                    f'WHERE {scope_sql(source,day,group,"bars")} AND resolution_ms=1000 '
                    f'AND bucket_index>={first} AND bucket_index<{last} GROUP BY ticker'))
            sizes={r['ticker']:int(r['n']) for r in counts}
            estimate=sum(sizes.values())*160/(1024**3)
            if estimate>MAX_PREPARED_GIB: raise MemoryError(f'Projected prepared data {estimate:.2f} GiB exceeds guard')
            groups=[]; group=[]; rows=0
            for ticker in names:
                n=sizes.get(ticker,0)
                if n>MAX_FETCH_ROWS: raise ValueError('Single listing exceeds fetch budget; shorten session explicitly')
                if group and (rows+n>MAX_FETCH_ROWS or len(group)>=MAX_FETCH_TICKERS):
                    groups.append(group); group=[]; rows=0
                group.append(ticker); rows+=n
            if group: groups.append(group)
            chunks=[]; frames=[]
            checkpoint=cache/'fetch-progress.json'
            progress=json.loads(checkpoint.read_text()) if checkpoint.is_file() else {'key':key,'chunks':{}}
            if progress['key']!=key: raise ValueError('Wrong partial-cache identity')
            for i,group in enumerate(groups):
                bs=scope_sql(source,day,group,'bars'); ts=scope_sql(source,day,group,'technical')
                bounds=f' AND resolution_ms=1000 AND bucket_index>={first} AND bucket_index<{last}'
                statement=(f'SELECT b.ticker AS ticker,toInt64(b.bucket_index) AS second,b.close_int/10000. AS close,'
                    f'toInt64(b.price_valid) AS price_valid,b.execution_volume AS volume,b.execution_notional AS notional,'
                    f'i.macd AS macd,i.signal AS signal,toInt64(i.seen) AS indicator_seen '
                    f'FROM (SELECT ticker,bucket_index,close_int,price_valid,execution_volume,execution_notional '
                    f'FROM arte.bars_v1 WHERE {bs}{bounds}) AS b LEFT JOIN '
                    f'(SELECT ticker,bucket_index,macd_line AS macd,macd_signal AS signal,1 AS seen '
                    f'FROM arte.indicators_v1 WHERE {ts}{bounds}) AS i '
                    f'ON b.ticker=i.ticker AND b.bucket_index=i.bucket_index ORDER BY second,ticker')
                expected=sum(sizes.get(t,0) for t in group)
                path=cache/f'chunk-{i:04d}.parquet'
                prior=progress['chunks'].get(str(i))
                if prior:
                    if prior['tickers']!=group or prior['rows']!=expected or file_hash(path)!=prior['sha256']:
                        raise ValueError('Partial chunk certificate mismatch')
                    frame=pl.read_parquet(path)
                    chunks.append({k:prior[k] for k in ('file','sha256','rows')}); frames.append(frame)
                    print({'resumed_chunk':i+1,'total_chunks':len(groups)},flush=True)
                    continue
                frame=arte_source.frame(reader,statement,DATA_SCHEMA)
                if frame.height!=expected or frame.select(pl.struct('ticker','second').n_unique()).item()!=expected:
                    raise ValueError('Fetched row count/key mismatch')
                if frame.filter((pl.col('price_valid')==1)&(pl.col('indicator_seen')!=1)).height:
                    raise ValueError('Bar without certified technical row')
                frame=frame.with_columns(pl.when(pl.col('indicator_seen')==1).then(pl.col('macd')).otherwise(None).alias('macd'),
                    pl.when(pl.col('indicator_seen')==1).then(pl.col('signal')).otherwise(None).alias('signal'))
                frame=frame.join(identities,on='ticker',how='left',validate='m:1').drop('ticker','indicator_seen')
                if frame['listing_id'].null_count(): raise ValueError('Unresolved historical identity')
                temporary=cache/f'chunk-{i:04d}.tmp'; frame.write_parquet(temporary); temporary.replace(path)
                item={'file':path.name,'sha256':file_hash(path),'rows':frame.height}
                chunks.append(item); frames.append(frame)
                progress['chunks'][str(i)]={**item,'tickers':group}
                temporary=cache/'fetch-progress.tmp'
                temporary.write_text(json.dumps(progress,indent=2)); temporary.replace(checkpoint)
                print({'fetched_chunks':i+1,'total_chunks':len(groups),'rows':frame.height},flush=True)
            market=pl.concat(frames)
            saved={'key':key,'chunks':chunks,'signature':signature,'population':population_proof,
                   'population_count':len(names),'zero_bar_listings':len(names)-len(sizes)}
            temp=cache/'complete.tmp'; temp.write_text(json.dumps(saved,default=str,indent=2)); temp.replace(cache_manifest)
    market=market.sort('second','listing_id')
    if market.estimated_size()/(1024**3)>MAX_PREPARED_GIB: raise MemoryError('Prepared data exceeds guard')
    invalid=market.filter((pl.col('price_valid')!=1)|(pl.col('close')<=0))
    if market.filter(~pl.all_horizontal(pl.col('close','volume','notional').is_finite()) |
                     (pl.col('price_valid')==1)&~pl.all_horizontal(pl.col('macd','signal').is_finite())).height:
        raise ValueError('Non-finite source price/liquidity/valid-price indicator')
    if market.filter((pl.col('volume')<0)|(pl.col('notional')<0)|
                     ((pl.col('volume')>0)&(pl.col('notional')<=0))).height:
        raise ValueError('Invalid execution volume/notional')
    market=market.with_columns(((pl.col('price_valid')==1)&(pl.col('close')>0)).alias('valid'),
        pl.when(pl.col('volume')>0).then(pl.col('notional')/pl.col('volume')).otherwise(None).alias('vwap'))
    print({'population':len(names),'session_rows':market.height,'slots':last-first,
           'zero_bar_listings':saved['zero_bar_listings'],'invalid_price_rows':invalid.height,
           'prepared_MiB':market.estimated_size()/1024**2,'preparation_seconds':perf_counter()-started},flush=True)
    return {'market':market,'first':first,'last':last,'population':identities,
            'source_key':key,'cache':str(cache),'preparation_seconds':perf_counter()-started}


## Strategy parameters and native expression rules

The searchable strategy is `Theta`, while participation, fees and the objective
weights belong to fixed experiment semantics. No optimizer is included. MACD
thresholds are normalized by price so different prices share a comparable rule.
Rules use completed columns; a submitted order cannot fill in its decision bar.
Stops/targets use completed close, then submit SELL for a following interval.
Partial BUY remainders carry until filled or cancelled by an exit; SELL has priority.
Cash is shared through deterministic proportional allocation, including actual
same-interval sale proceeds. There is no order-size reservation, as in the prior
Polars contract. Keep this choice fixed when comparing objectives.


In [ ]:
@dataclass(frozen=True)
class Theta:
    min_price:float=1.0
    max_price:float=50.0
    entry_gap_bps:float=2.0
    exit_gap_bps:float=-1.0
    min_execution_volume:float=100.0
    entry_budget:float=2_000.0
    stop_fraction:float=0.02
    target_fraction:float=0.04

OBJECTIVE_DRAWDOWN_WEIGHT=0.0  # Set once for the experiment, not inside Theta.

def prepare_policy(prepared,theta):
    if not all(math.isfinite(v) for v in asdict(theta).values()): raise ValueError('Non-finite theta')
    if not (0<theta.min_price<theta.max_price and theta.entry_budget>0 and
            theta.min_execution_volume>=0 and 0<theta.stop_fraction<1 and theta.target_fraction>0):
        raise ValueError('Invalid strategy bounds')
    m=prepared['market'].with_columns(((pl.col('macd')-pl.col('signal'))/pl.col('close')*10000).alias('gap_bps'))
    rules={
        'entry':pl.col('valid') & pl.col('close').is_between(theta.min_price,theta.max_price) &
            (pl.col('gap_bps')>theta.entry_gap_bps)&(pl.col('macd')>0)&
            (pl.col('volume')>=theta.min_execution_volume),
        'exit':pl.col('valid') & (pl.col('gap_bps')<theta.exit_gap_bps),
    }
    # Stateless work vectorized across the whole session; no future values enter expressions.
    m=m.with_columns(rules['entry'].fill_null(False).alias('entry'),rules['exit'].fill_null(False).alias('exit'))
    return {int(clock[0]):frame for clock,frame in m.partition_by('second',as_dict=True,maintain_order=True).items()}

STATE_SCHEMA={'listing_id':pl.String,'quantity':pl.Int64,'book_cost':pl.Float64,
              'side':pl.Int64,'remaining':pl.Int64,'stop':pl.Float64,'target':pl.Float64,
              'mark':pl.Float64,'mark_second':pl.Int64}
STATE_COLUMNS=list(STATE_SCHEMA)

def evaluate(theta,prepared,keep_fills=False,progress=False):
    started=perf_counter(); bars=prepare_policy(prepared,theta); policy_seconds=perf_counter()-started
    state=pl.DataFrame(schema=STATE_SCHEMA); cash=INITIAL_CASH; realized=0.0
    accounts=[]; fill_frames=[]; total_filled=0; transitions=0; fee_total=0.0
    empty=prepared['market'].head(0).with_columns(pl.lit(False).alias('entry'),pl.lit(False).alias('exit'))
    for second in range(prepared['first'],prepared['last']):
        if progress and (second-prepared['first'])%3600==0:
            print({'completed_slots':second-prepared['first'],'total_slots':prepared['last']-prepared['first']},flush=True)
        current=bars.get(second,empty)
        # Preserve all active positions/orders, but avoid evaluating unrelated inactive rows.
        selected=current.filter(pl.col('entry')|pl.col('listing_id').is_in(state['listing_id'].implode()))
        if state.is_empty() and selected.is_empty():
            accounts.append((second+1,cash,realized,cash,0.0,0)); continue
        transitions+=1
        rows=state.lazy().join(selected.lazy(),on='listing_id',how='full',coalesce=True).with_columns(
            pl.col('quantity','remaining','side').fill_null(0),
            pl.col('book_cost','stop','target','mark').fill_null(0.0),pl.col('mark_second').fill_null(-1),
            pl.col('entry','exit','valid').fill_null(False),pl.col('volume').fill_null(0.0),
        ).with_columns(
            pl.when(pl.col('valid')).then(pl.col('close')).otherwise(pl.col('mark')).alias('mark'),
            pl.when(pl.col('valid')).then(second+1).otherwise(pl.col('mark_second')).alias('mark_second'),
            pl.when(pl.col('quantity')>0).then(pl.col('book_cost')/pl.col('quantity')).otherwise(0.0).alias('average'),
            pl.when(pl.col('valid')&(pl.col('volume')>0)&pl.col('vwap').is_not_null())
              .then((pl.col('volume')*PARTICIPATION).floor()).otherwise(0).cast(pl.Int64).alias('capacity'),
        ).with_columns(
            pl.when(pl.col('side')==-1).then(pl.min_horizontal('quantity','remaining','capacity'))
              .otherwise(0).alias('sell'),
            pl.when(pl.col('side')==1).then(pl.min_horizontal('remaining','capacity')).otherwise(0).alias('buy_candidate'),
        ).with_columns(
            (pl.col('sell')*pl.col('vwap').fill_null(0.0)*(1-FEE_BPS/10000)).alias('sale_cash'),
            (pl.col('buy_candidate')*pl.col('vwap').fill_null(0.0)*(1+FEE_BPS/10000)).alias('buy_need'),
        ).collect()
        totals=rows.select(pl.col('sale_cash').sum(),pl.col('buy_need').sum()).row(0)
        spendable=cash+totals[0]; scale=min(1.0,spendable/totals[1]) if totals[1]>0 else 1.0
        rows=rows.lazy().with_columns((pl.col('buy_candidate')*scale).floor().cast(pl.Int64).alias('buy')).with_columns(
            (pl.col('quantity')+pl.col('buy')-pl.col('sell')).alias('post_quantity'),
            (pl.col('remaining')-pl.col('buy')-pl.col('sell')).alias('post_remaining'),
            (pl.col('buy')*pl.col('vwap').fill_null(0.0)*FEE_BPS/10000+
             pl.col('sell')*pl.col('vwap').fill_null(0.0)*FEE_BPS/10000).alias('fee'),
            (pl.col('sale_cash')-pl.col('buy')*pl.col('vwap').fill_null(0.0)*(1+FEE_BPS/10000)).alias('cash_delta'),
            (pl.col('sell')*(pl.col('vwap').fill_null(0.0)*(1-FEE_BPS/10000)-pl.col('average'))).alias('realized_delta'),
            (pl.col('book_cost')+pl.col('buy')*pl.col('vwap').fill_null(0.0)*(1+FEE_BPS/10000)
             -pl.col('sell')*pl.col('average')).alias('post_cost'),
        ).with_columns(
            pl.when((pl.col('quantity')==0)&(pl.col('buy')>0)).then(pl.col('vwap')*(1-theta.stop_fraction))
              .otherwise(pl.col('stop')).alias('stop'),
            pl.when((pl.col('quantity')==0)&(pl.col('buy')>0)).then(pl.col('vwap')*(1+theta.target_fraction))
              .otherwise(pl.col('target')).alias('target'),
        )
        # Existing pending sells take precedence; exits cancel any outstanding buy remainder.
        rows=rows.with_columns(((pl.col('post_quantity')>0)&pl.col('valid')&
            (pl.col('exit')|(pl.col('mark')<=pl.col('stop'))|(pl.col('mark')>=pl.col('target')))).alias('request_exit'),
            ((pl.col('post_quantity')==0)&(pl.col('post_remaining')==0)&pl.col('entry')).alias('request_entry'))
        rows=rows.with_columns(
            pl.when((pl.col('side')==-1)&(pl.col('post_remaining')>0)).then(-1)
              .when(pl.col('request_exit')).then(-1).when(pl.col('request_entry')).then(1)
              .when(pl.col('post_remaining')>0).then(pl.col('side')).otherwise(0).cast(pl.Int64).alias('next_side'),
            pl.when(pl.col('request_exit')).then(pl.col('post_quantity'))
              .when(pl.col('request_entry')).then((theta.entry_budget/pl.col('mark')).floor())
              .otherwise(pl.col('post_remaining')).cast(pl.Int64).alias('next_remaining'),
        ).collect()
        sums=rows.select(pl.col('cash_delta').sum(),pl.col('realized_delta').sum(),pl.col('fee').sum(),
                         (pl.col('buy')+pl.col('sell')).sum()).row(0)
        cash+=sums[0]; realized+=sums[1]; fee_total+=sums[2]; total_filled+=sums[3]
        if cash < -1e-7: raise AssertionError('Cash allocation violated')
        if keep_fills:
            fill_frames.append(rows.filter((pl.col('buy')+pl.col('sell'))>0).select('listing_id',
                pl.lit(second+1).alias('boundary_second'),'buy','sell','capacity','vwap','cash_delta','fee'))
        state=rows.lazy().filter((pl.col('post_quantity')>0)|(pl.col('next_remaining')>0)).select('listing_id',
            pl.col('post_quantity').alias('quantity'),pl.col('post_cost').alias('book_cost'),
            pl.col('next_side').alias('side'),pl.col('next_remaining').alias('remaining'),
            'stop','target','mark','mark_second').sort('listing_id').collect()
        mv,stale=state.select((pl.col('quantity')*pl.col('mark')).sum().alias('market_value'),
            ((pl.col('quantity')>0)&(pl.col('mark_second')<second+1)).sum().alias('stale')).row(0)
        accounts.append((second+1,cash,realized,cash+mv,mv,stale))
    account=pl.DataFrame(accounts,schema=['boundary_second','cash','realized_pnl','equity','market_value','stale_held_marks'],orient='row')
    account=account.with_columns(pl.col('equity').cum_max().clip(lower_bound=INITIAL_CASH).alias('peak'))
    summary=account.select((pl.col('equity').last()/INITIAL_CASH-1).alias('return_fraction'),
        (1-pl.col('equity')/pl.col('peak')).max().alias('max_drawdown'),pl.col('stale_held_marks').max().alias('max_stale_held_marks')).row(0,named=True)
    objective=summary['return_fraction']-OBJECTIVE_DRAWDOWN_WEIGHT*summary['max_drawdown']
    # Terminal positions remain marked; do not fabricate instantaneous liquidation.
    return dict(objective=objective,**summary,accounts=account,state=state,
        filled_shares=total_filled,fees=fee_total,transition_slots=transitions,
        policy_seconds=policy_seconds,wall_seconds=perf_counter()-started,
        fills=pl.concat(fill_frames) if fill_frames else pl.DataFrame())


## Independent correctness checks before real execution

This bounded fixture proves causal delay, zero-volume nonfills, participation
capacity, fee-aware cash allocation, marked-equity accounting, and future-tail
invariance. It is a test fixture only: it never substitutes for the real-data run.


In [ ]:
def fixture():
    m=pl.DataFrame({'listing_id':['A']*5,'second':list(range(5)),
        'close':[10.0]*5,'price_valid':[1]*5,'volume':[1000.,0.,20.,20.,20.],
        'notional':[10000.,0.,200.,200.,200.], 'macd':[0.1]*5,'signal':[0.0]*5})
    m=m.with_columns(pl.lit(True).alias('valid'),
        pl.when(pl.col('volume')>0).then(pl.col('notional')/pl.col('volume')).otherwise(None).alias('vwap'))
    return {'market':m,'first':0,'last':5}
f=fixture(); theta=Theta(entry_budget=100.,min_execution_volume=0.)
a=evaluate(theta,f,keep_fills=True)
assert a['fills']['boundary_second'].to_list()==[3,4,5]
assert a['fills'].select(((pl.col('buy')+pl.col('sell'))<=pl.col('capacity')).all()).item()
assert a['state']['quantity'].sum()==a['fills'].select((pl.col('buy')-pl.col('sell')).sum()).item()
assert abs(a['accounts']['cash'][-1]-INITIAL_CASH-a['fills']['cash_delta'].sum())<1e-8
assert abs(a['accounts']['equity'][-1]-INITIAL_CASH-a['accounts']['realized_pnl'][-1]-
           (a['state'].select((pl.col('quantity')*pl.col('mark')-pl.col('book_cost')).sum()).item() or 0))<1e-8
changed=dict(f); changed['market']=f['market'].with_columns(
    pl.when(pl.col('second')>=3).then(pl.col('close')*2).otherwise(pl.col('close')).alias('close'))
b=evaluate(theta,changed,keep_fills=True)
assert a['accounts'].filter(pl.col('boundary_second')<=3).equals(b['accounts'].filter(pl.col('boundary_second')<=3))
# A close-triggered exit cancels the unfinished BUY and first sells next interval.
exiting=dict(f); exiting['market']=f['market'].with_columns(
    pl.when(pl.col('second')>=3).then(9.0).otherwise(pl.col('close')).alias('close'))
x=evaluate(theta,exiting,keep_fills=True)
assert x['fills'].filter(pl.col('sell')>0)['boundary_second'].to_list()==[5]
assert x['state']['side'].item()==-1 and x['state']['quantity'].item()==2
assert x['state']['remaining'].item()==2
# Competing orders use shared cash, including fees, with symmetric pro-rata sizing.
original_cash=INITIAL_CASH
try:
    INITIAL_CASH=100.0
    scarce=dict(f); scarce['market']=pl.concat([f['market'],f['market'].with_columns(pl.lit('B').alias('listing_id'))])
    y=evaluate(theta,scarce,keep_fills=True)
    assert y['accounts']['cash'].min()>=-1e-8
    assert y['state']['quantity'].to_list()==[4,4]
    assert abs(y['accounts']['cash'][-1]-100.0-y['fills']['cash_delta'].sum())<1e-8
finally:
    INITIAL_CASH=original_cash
print('PASS: delayed/partial fills, zero volume, capacity, conservation, fees, shared cash, exit cancellation and future tail')


## Real-session preparation and measured objective evaluation

Cold preparation reports certification/fetch/cache time; warm preparation verifies
cached file hashes and source identity. Objective timing includes full-session
rule calculation, frame partitioning, replay and result assembly; no database I/O.
Compact account history is retained (19,800 rows), detailed fills are optional.
Repeat the same theta to verify determinism and report actual timings. No parameter
search, training, source writers or services are launched.


In [ ]:
prepared=prepare_session()


In [ ]:
theta=Theta()
result=evaluate(theta,prepared,progress=True)
print({k:v for k,v in result.items() if k not in ('accounts','state','fills')})
print(result['accounts'].tail(5))
print({'time_slots':prepared['last']-prepared['first'],'population':prepared['population'].height,
       'rows':prepared['market'].height,'open_positions':result['state'].filter(pl.col('quantity')>0).height,
       'pending_orders':result['state'].filter(pl.col('remaining')>0).height})
repeat=evaluate(theta,prepared)
assert result['accounts'].equals(repeat['accounts']) and result['state'].equals(repeat['state'])
assert result['objective']==repeat['objective']
print({'repeat_wall_seconds':repeat['wall_seconds'],'deterministic':True,
       'simulated_seconds_per_wall_second':(prepared['last']-prepared['first'])/repeat['wall_seconds']})


## Measured August 18 premarket example

Validated on the laptop with Polars 1.40.1 on 2026-10-01. The fixed certified
population contains 6,100 listings, of which 4,843 have at least one premarket bar.
12 fetch chunks returned 770,851 rows, including 505,188 invalid-price rows; no
listings were selected by profitability. Prepared data occupies about 84.4 MiB.
Cold certification/fetch/cache took **27.27 s**; verified warm-cache preparation
**6.69 s**. These preparation costs are outside repeated strategy evaluation.

Full 04:00–09:30 replay contains **19,800 one-second slots**. Initial eager
24-thread transitions took **120.99 s**. Fused lazy transitions with one Polars
thread took **43.00 s**, then **40.07 s** on repeat, including whole-session rules,
partitioning, account replay and compact result assembly. Rule/partition work was
**0.264 s** in the first optimized run. Combined optimization improved this case
about 3×; it does not prove absolute maximum speed on other hardware or strategies.
Start a fresh kernel to apply `POLARS_MAX_THREADS`; the notebook reports the actual
thread count, and respects an explicit environment override.

Baseline and optimized runs matched return, drawdown, fees and filled-share totals;
the two optimized runs had identical full account history and final state. This
strategy left **256 open positions and 480 pending orders**, and carried **216
held marks without an update in the final second**. They are displayed terminal
exposure, not liquidated results. Strategy profitability and realistic broker
acceptance have not been established; search is intentionally the next step.

## How the next search stage should call this

Keep `prepared` and the fixed environment/objective definition unchanged, then call
`evaluate(Theta(...), prepared)['objective']` for each candidate. No network fetch
is in that objective. In later profiling, amortize repeated partitioning and
parameter-independent market columns only after measuring their share. Parallel
candidates require bounded processes and memory; do not multiply server readers.

This notebook measures one real premarket session, not general maximum speed.
Report dataset shape, Polars version/threads, cold preparation and warm evaluation
separately. Session-marked returns include exposure and stale-mark risk at the end;
search acceptance needs a fixed terminal treatment, realistic costs and sealed
out-of-sample dates. The notebook exposes open/pending states instead of hiding them.
A favorable objective on August 18 is implementation evidence, not strategy acceptance.
